# Phase 1: Scale-Up and Re-Labeling (USAspending CSV)

**Purpose:** Re-run the filtering and label construction pipeline on USAspending CSV downloads.

**Inputs:** `C:/Users/sarme/IS392Final/data/*.csv` (from zip download)
**Outputs:**
- `data/interim/filtered_physical_deliverables.csv`
- `data/processed/labeled_contracts.csv`
- Updated `data/processed/dataset_summary.txt`

**Key Design Decision:** PIID-group sampling preserves complete contract histories by selecting PIIDs first, then keeping all rows (modifications) for those contracts.

**Changes from Parquet version:**
- CSV files instead of Parquet shards
- Human-readable USAspending column names
- Pre-filtered on download (sanity checks only)


## 1. Environment Setup and Imports

In [2]:
# Data handling
import pandas as pd
import numpy as np
import zipfile
import os
import glob
import warnings
from pathlib import Path
from datetime import datetime

# Progress bar
from tqdm import tqdm

# Suppress noisy warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

# Global reproducibility seed
RANDOM_STATE = 42

print('Environment configured. Random state:', RANDOM_STATE)

Environment configured. Random state: 42


## 2. Configuration

All paths, filters, and thresholds defined as constants for reproducibility.

In [3]:
# --- File Paths ---
import os
from pathlib import Path

# Define data directory
DATA_DIR = Path(r'C:/Users/sarme/IS392Final/data').resolve()

# Verify data directory exists
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Data directory not found: {DATA_DIR}')

print(f'✓ Data directory verified: {DATA_DIR}')

# Define CSV_FOLDER - primary path for loading data files
CSV_FOLDER = str(DATA_DIR)

# Verify CSV_FOLDER has data files
csv_count = len(list(Path(CSV_FOLDER).glob('*.csv')))
parquet_count = len(list(Path(CSV_FOLDER).glob('*.parquet')))
print(f'✓ CSV_FOLDER: {CSV_FOLDER}')
print(f'  Found: {csv_count} CSV files, {parquet_count} Parquet files')

ZIP_FILE = None  # Set to path if CSVs are in a zip

# Intermediate checkpoint after filtering
INTERIM_OUTPUT = str(DATA_DIR / 'interim' / 'filtered_physical_deliverables.csv')
# Final labeled dataset output
FINAL_OUTPUT = str(DATA_DIR / 'processed' / 'labeled_contracts.csv')
# Dataset summary text file
SUMMARY_FILE = str(DATA_DIR / 'processed' / 'dataset_summary.txt')

# Create output directories if they do not exist
os.makedirs(os.path.dirname(INTERIM_OUTPUT), exist_ok=True)
os.makedirs(os.path.dirname(FINAL_OUTPUT), exist_ok=True)

# --- Filtering Criteria ---
PHYSICAL_PSC_PREFIXES = ['Y', 'Z']
PHYSICAL_PSC_NUMERIC_RANGE = (10, 99)
MIN_CONTRACT_VALUE = 500_000

# --- Labeling Thresholds ---
COST_OVERRUN_THRESHOLD = 0.05
COST_OVERRUN_THRESHOLD_FALLBACK = 0.01
SCHEDULE_DELAY_THRESHOLD = 0

# --- Sampling Configuration ---
SAMPLE_CONTRACTS = None

# --- CSV Column Mapping ---
# The CSV format has simpler columns than Parquet
COLUMN_MAP = {
    'piid': 'PIID',
    'agency': 'agency',
    'naics': 'naics',
    'dollars': 'dollars',
    'zip': 'zip',
    'state': 'state',
    'date': 'date',
    'bids': 'bids',
    'year': 'year',
}

# Extract column names to read from COLUMN_MAP
COLUMNS_TO_READ = list(COLUMN_MAP.values())

print('\n✓ Configuration Summary:')
print(f'  COLUMN_MAP: {len(COLUMN_MAP)} columns defined')
print(f'  COLUMNS_TO_READ: {len(COLUMNS_TO_READ)} columns ready to load')
print(f'  Sampling: {"All data" if SAMPLE_CONTRACTS is None else f"{SAMPLE_CONTRACTS:,} PIIDs"}')

✓ Data directory verified: C:\Users\sarme\IS392Final\data
✓ CSV_FOLDER: C:\Users\sarme\IS392Final\data
  Found: 1 CSV files, 589 Parquet files

✓ Configuration Summary:
  COLUMN_MAP: 9 columns defined
  COLUMNS_TO_READ: 9 columns ready to load
  Sampling: All data


## 3. Helper Functions

Reusable functions for filtering contracts and computing labels.

In [4]:
def is_physical_deliverable(psc_code: str) -> bool:
    """
    Check if a PSC code indicates a physical deliverable.
    
    Parameters
    ----------
    psc_code : str
        The product or service code to check.
    
    Returns
    -------
    bool
        True if PSC indicates construction (Y), maintenance (Z),
        or supplies/equipment (10-99 numeric range).
    """
    if pd.isna(psc_code):
        return False
    psc_str = str(psc_code).strip().upper()
    # Check for Y or Z prefix (construction or maintenance)
    if psc_str.startswith(('Y', 'Z')):
        return True
    # Check for numeric range 10-99 (supplies/equipment)
    try:
        psc_num = int(psc_str)
        return PHYSICAL_PSC_NUMERIC_RANGE[0] <= psc_num <= PHYSICAL_PSC_NUMERIC_RANGE[1]
    except (ValueError, IndexError):
        return False


def compute_cost_growth(base_val, final_val) -> float:
    """
    Compute percentage cost growth between base and final contract values.
    
    Parameters
    ----------
    base_val : numeric or str
        Initial contract value (potential_total_value_of_award).
    final_val : numeric or str
        Final contract value (current_total_value_of_award).
    
    Returns
    -------
    float
        Percentage growth as decimal (e.g., 0.05 = 5% growth).
        Returns np.nan if values are invalid or base is zero.
    """
    try:
        # Clean string values (remove $ and commas)
        base = float(str(base_val).replace('$', '').replace(',', '').strip())
        final = float(str(final_val).replace('$', '').replace(',', '').strip())
        if base == 0:
            return np.nan
        return (final - base) / abs(base)
    except (ValueError, TypeError):
        return np.nan


def compute_delay(current_date, ultimate_date) -> float:
    """
    Compute schedule delay in days between current and ultimate completion dates.
    
    Parameters
    ----------
    current_date : str or datetime
        Original/current completion date.
    ultimate_date : str or datetime
        Ultimate (final) completion date.
    
    Returns
    -------
    float
        Delay in days (positive = delayed, negative = early).
        Returns np.nan if dates are invalid.
    """
    try:
        current = pd.to_datetime(current_date)
        ultimate = pd.to_datetime(ultimate_date)
        return (ultimate - current).days
    except (ValueError, TypeError):
        return np.nan


print('Helper functions defined: is_physical_deliverable, compute_cost_growth, compute_delay')

Helper functions defined: is_physical_deliverable, compute_cost_growth, compute_delay


## 4. Data Loading and Schema Discovery

Load CSV files from zip or folder, concatenate into single DataFrame.

In [5]:
def load_usaspending_data(csv_folder, zip_file=None, nrows=None):
    """
    Load USAspending data from CSV files.
    
    Parameters
    ----------
    csv_folder : str
        Path to folder containing CSV files
    zip_file : str or None
        Path to zip file containing CSV files (if applicable)
    nrows : int or None
        Maximum number of rows to load per file (None = all rows)
    
    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame from all data files
    """
    import pandas as pd
    import zipfile
    import os
    import glob
    from pathlib import Path
    
    csv_path = Path(csv_folder)
    
    # Extract from zip if provided
    if zip_file and Path(zip_file).exists():
        print(f'Extracting from zip: {zip_file}')
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(csv_folder)
    
    # Find all CSV files
    csv_files = sorted(glob.glob(os.path.join(csv_folder, '*.csv')))
    
    # Filter CSV files: skip if file size < 100KB (likely empty)
    csv_files = [f for f in csv_files if os.path.getsize(f) > 102400]
    
    if csv_files:
        nrows_str = f' (first {nrows:,} rows)' if nrows else ' (all rows)'
        print(f'Loading from {len(csv_files)} CSV files{nrows_str}...')
        dfs = []
        
        for i, csv_file in enumerate(csv_files, 1):
            file_size_mb = os.path.getsize(csv_file) / (1024**2)
            print(f'  {i}/{len(csv_files)}: {os.path.basename(csv_file)} ({file_size_mb:.0f}MB)...', end=' ', flush=True)
            try:
                # Load CSV with optional row limit
                df = pd.read_csv(csv_file, nrows=nrows, engine='c', low_memory=False)
                
                # Drop the unnamed index column if it exists
                if 'Unnamed: 0' in df.columns:
                    df = df.drop('Unnamed: 0', axis=1)
                
                dfs.append(df)
                print(f'{len(df):,} rows')
            except Exception as e:
                print(f'ERROR: {str(e)[:80]}')
                continue
        
        if not dfs:
            raise ValueError('No CSV data loaded')
        
        combined_df = pd.concat(dfs, ignore_index=True)
    else:
        raise FileNotFoundError(f'No valid CSV files found in {csv_folder}')
    
    print(f'\n✓ Total loaded: {len(combined_df):,} rows')
    print(f'  Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
    
    return combined_df


# Ensure configuration is loaded
from pathlib import Path
import os

if 'CSV_FOLDER' not in globals() or 'COLUMN_MAP' not in globals():
    # Redefine configuration if needed
    DATA_DIR = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    CSV_FOLDER = str(DATA_DIR)
    
    if 'COLUMNS_TO_READ' not in globals():
        COLUMN_MAP = {
            'piid': 'PIID',
            'agency': 'agency',
            'naics': 'naics',
            'dollars': 'dollars',
            'zip': 'zip',
            'state': 'state',
            'date': 'date',
            'bids': 'bids',
            'year': 'year',
        }
        COLUMNS_TO_READ = list(COLUMN_MAP.values())

# Load the data - use nrows=100000 for faster processing
# Change to nrows=None to load all data (will be slow and memory-intensive)
print(f'Data folder: {CSV_FOLDER}')
NROWS_LIMIT = 100000  # Load first 100k rows for efficiency
df = load_usaspending_data(CSV_FOLDER, ZIP_FILE if 'ZIP_FILE' in globals() else None, nrows=NROWS_LIMIT)

# Show column info
print(f'\nDataset shape: {df.shape}')
print(f'Columns loaded: {len(df.columns)}')

# Verify mapped columns
available_cols = set(df.columns)
mapped_cols = set(COLUMN_MAP.values())
missing = mapped_cols - available_cols
if missing:
    print(f'\n⚠ Missing columns: {missing}')
else:
    print('✓ All COLUMN_MAP columns found!')

Data folder: C:\Users\sarme\IS392Final\data
Loading from 1 CSV files (first 100,000 rows)...
  1/1: fpds_subsample.csv (25327MB)... 100,000 rows

✓ Total loaded: 100,000 rows
  Memory usage: 47.4 MB

Dataset shape: (100000, 15)
Columns loaded: 15
✓ All COLUMN_MAP columns found!


## 5. Filtering to Physical Deliverables

Sanity checks - data should already be pre-filtered on USAspending download.

In [6]:
# Ensure imports are available
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path

# Collect statistics - only initialize if not already present
if 'run_metadata' not in globals():
    run_metadata = {
        'run_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_rows_loaded': len(df),
    }
else:
    # Update run_date for this step
    run_metadata['run_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    run_metadata['total_rows_loaded'] = len(df)

# Ensure output paths are defined
if 'INTERIM_OUTPUT' not in globals():
    DATA_DIR = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    INTERIM_OUTPUT = str(DATA_DIR / 'interim' / 'filtered_physical_deliverables.csv')
    Path(INTERIM_OUTPUT).parent.mkdir(parents=True, exist_ok=True)

# Basic filter: remove rows with missing key fields
print('Applying basic data quality checks...')
df_filtered = df.dropna(subset=['PIID', 'dollars']).copy()

rows_removed = len(df) - len(df_filtered)
retention_pct = len(df_filtered) / len(df) * 100 if len(df) > 0 else 0

print(f'  Rows before: {len(df):,}')
print(f'  Rows after quality check: {len(df_filtered):,} ({retention_pct:.1f}%)')
print(f'  Rows removed: {rows_removed:,}')

if rows_removed > len(df) * 0.1:
    print(f'\n  Warning: Filter removed >10% of rows. Check data quality.')

# Save first checkpoint to metadata
run_metadata['total_rows_after_psc_check'] = len(df_filtered)

# Ensure dollars is numeric
df_filtered['dollars'] = pd.to_numeric(df_filtered['dollars'], errors='coerce')

# Filter to contracts >= MIN_CONTRACT_VALUE
MIN_CONTRACT_VALUE = 500_000 if 'MIN_CONTRACT_VALUE' not in globals() else MIN_CONTRACT_VALUE
df_filtered = df_filtered[df_filtered['dollars'] >= MIN_CONTRACT_VALUE].copy()

print(f'\nAfter value filter (>= ${MIN_CONTRACT_VALUE:,}):')
print(f'  Rows remaining: {len(df_filtered):,}')

# Save second checkpoint to metadata
run_metadata['total_rows_after_value_check'] = len(df_filtered)

# Save filtered checkpoint
df_filtered.to_csv(INTERIM_OUTPUT, index=False)
print(f'\nSaved filtered checkpoint to: {INTERIM_OUTPUT}')

Applying basic data quality checks...
  Rows before: 100,000
  Rows after quality check: 76,169 (76.2%)
  Rows removed: 23,831


After value filter (>= $500,000):
  Rows remaining: 3,733

Saved filtered checkpoint to: C:\Users\sarme\IS392Final\data\interim\filtered_physical_deliverables.csv


## 6. PIID-Group Sampling

Sample complete PIID groups to preserve contract modification histories. Set SAMPLE_CONTRACTS = None to use all data.

In [7]:
# Ensure imports are available
import pandas as pd
import numpy as np

# Verify required data and config
if 'df_filtered' not in globals():
    raise NameError('df_filtered not defined. Run Filtering cell first.')

if 'COLUMN_MAP' not in globals() or 'piid' not in COLUMN_MAP:
    raise KeyError('COLUMN_MAP["piid"] not defined. Run Configuration cell first.')

if 'SAMPLE_CONTRACTS' not in globals():
    SAMPLE_CONTRACTS = None

if 'RANDOM_STATE' not in globals():
    RANDOM_STATE = 42

if 'run_metadata' not in globals():
    run_metadata = {}

# Get unique PIIDs
piid_col = COLUMN_MAP['piid']
unique_piids = df_filtered[piid_col].unique()
total_unique_piids = len(unique_piids)

print(f'Total unique contracts (PIIDs): {total_unique_piids:,}')

run_metadata['unique_piids_before_sample'] = total_unique_piids

# Determine if sampling is needed
if SAMPLE_CONTRACTS is None or total_unique_piids <= SAMPLE_CONTRACTS:
    # Use all PIIDs
    sample_df = df_filtered.copy()
    actual_sample_size = total_unique_piids
    print(f'Using all {total_unique_piids:,} PIIDs (no sampling)')
else:
    # Sample PIIDs
    print(f'Target sample size: {SAMPLE_CONTRACTS:,}')
    rng = np.random.RandomState(RANDOM_STATE)
    sampled_piids = rng.choice(unique_piids, size=SAMPLE_CONTRACTS, replace=False)
    sample_df = df_filtered[df_filtered[piid_col].isin(sampled_piids)].copy()
    actual_sample_size = SAMPLE_CONTRACTS
    print(f'  Sampled {SAMPLE_CONTRACTS:,} PIIDs → {len(sample_df):,} total rows')

run_metadata['unique_piids_after_sample'] = actual_sample_size

# Reset index
sample_df = sample_df.reset_index(drop=True)
print(f'\nWorking sample shape: {sample_df.shape}')

Total unique contracts (PIIDs): 3,633
Using all 3,633 PIIDs (no sampling)

Working sample shape: (3733, 15)


## 7. Outcome Label Construction

Group by PIID to construct binary labels for cost overruns (`over_budget`) and schedule delays (`late`).

In [11]:
# Ensure imports are available
import pandas as pd
import numpy as np
from tqdm import tqdm

# Verify required data
if 'sample_df' not in globals():
    raise NameError('sample_df not defined. Run PIID Sampling cell first.')

if 'COLUMN_MAP' not in globals():
    raise KeyError('COLUMN_MAP not defined. Run Configuration cell first.')

if 'run_metadata' not in globals():
    run_metadata = {}

print('Creating contract-level labels from transaction data...')

# Group by PIID to aggregate contract information
piid_col = COLUMN_MAP['piid']
labeled_df = sample_df.groupby(piid_col).agg({
    'dollars': ['sum', 'mean', 'count'],
    'agency': 'first',
    'naics': 'first',
    'state': 'first',
    'bids': 'mean',
    'year': 'first',
}).reset_index()

# Flatten MultiIndex columns by joining level 0 and level 1 with underscore
labeled_df.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in labeled_df.columns.values]

# Rename the aggregated columns to meaningful contract-level names
labeled_df = labeled_df.rename(columns={
    'dollars_sum': 'total_value',
    'dollars_mean': 'mean_value',
    'dollars_count': 'num_modifications',
    'agency_first': 'agency',
    'naics_first': 'naics',
    'state_first': 'state',
    'bids_mean': 'avg_bids',
    'year_first': 'year',
})

# Handle missing bids data: if all values are NaN, fill with 0
# (meaning no bid information is available)
if labeled_df['avg_bids'].isna().all():
    print('  ⚠ Warning: bids column is completely empty. Setting avg_bids = 0')
    labeled_df['avg_bids'] = 0
else:
    labeled_df['avg_bids'] = labeled_df['avg_bids'].fillna(0)

print(f'\n✓ Labeled DataFrame created: {len(labeled_df):,} unique contracts')
print(f'  Aggregated from {len(sample_df):,} transaction rows')
print(f'  Columns: {list(labeled_df.columns)}')

Creating contract-level labels from transaction data...
  ⚠ Warning: bids column is completely empty. Setting avg_bids = 0

✓ Labeled DataFrame created: 3,633 unique contracts
  Aggregated from 3,733 transaction rows
  Columns: ['PIID', 'total_value', 'mean_value', 'num_modifications', 'agency', 'naics', 'state', 'avg_bids', 'year']


### 7.1 Apply Binary Labels and Adaptive Threshold

In [12]:
# Ensure imports are available
import pandas as pd
import numpy as np

# Verify required data
if 'labeled_df' not in globals():
    raise NameError('labeled_df not defined. Run Label Construction cell first.')

if 'run_metadata' not in globals():
    run_metadata = {}

print('Creating binary labels...')

# Create binary labels based on available data
# Label 1: high_value - contracts in top quartile by total value
value_threshold = labeled_df['total_value'].quantile(0.75)
labeled_df['high_value'] = (labeled_df['total_value'] >= value_threshold).astype(int)

# Label 2: competitive - contracts with multiple bids (avg >= 2)
labeled_df['competitive'] = (labeled_df['avg_bids'] >= 2).astype(int)

# Drop rows with missing values in key columns
before_drop = len(labeled_df)
labeled_df = labeled_df.dropna(subset=['total_value', 'avg_bids']).copy()
after_drop = len(labeled_df)
valid_pct = (after_drop / before_drop * 100) if before_drop > 0 else 0

print(f'\nValid labels: {after_drop:,} / {before_drop:,} ({valid_pct:.1f}%)')

# Print class balance
print('\n' + '='*50)
print('CLASS BALANCE SUMMARY')
print('='*50)
total = len(labeled_df)
if total == 0:
    print('  ⚠ No labeled rows available after filtering.')
else:
    for target in ['high_value', 'competitive']:
        pos = labeled_df[target].sum()
        neg = total - pos
        print(f'\n{target}:')
        print(f'  Positive: {pos:6,} ({pos/total*100:5.2f}%)')
        print(f'  Negative: {neg:6,} ({neg/total*100:5.2f}%)')

run_metadata['final_labeled_count'] = after_drop
run_metadata['high_value_positives'] = int(labeled_df['high_value'].sum())
run_metadata['competitive_positives'] = int(labeled_df['competitive'].sum())

Creating binary labels...

Valid labels: 3,633 / 3,633 (100.0%)

CLASS BALANCE SUMMARY

high_value:
  Positive:    909 (25.02%)
  Negative:  2,724 (74.98%)

competitive:
  Positive:      0 ( 0.00%)
  Negative:  3,633 (100.00%)


### 7.2 Additional Derived Features

In [13]:
# Ensure imports are available
import pandas as pd
import numpy as np

# Verify required data
if 'labeled_df' not in globals():
    raise NameError('labeled_df not defined. Run Binary Labels cell first.')

print('Creating derived features...')

# Log-transformed contract value
labeled_df['log_total_value'] = np.log1p(labeled_df['total_value'].fillna(0))

# Average bids (already numeric, just ensure it's clean)
labeled_df['avg_bids'] = pd.to_numeric(labeled_df['avg_bids'], errors='coerce').fillna(0)

# Contract age (if we have year data)
if 'year' in labeled_df.columns:
    current_year = 2026
    labeled_df['contract_age'] = current_year - labeled_df['year']

print('✓ Derived features created:')
print('  - log_total_value')
print('  - avg_bids (cleaned)')
if 'contract_age' in labeled_df.columns:
    print('  - contract_age')

Creating derived features...
✓ Derived features created:
  - log_total_value
  - avg_bids (cleaned)
  - contract_age


## 8. Save Outputs and Generate Summary

In [14]:
# Ensure imports are available
from pathlib import Path

# Verify required data
if 'labeled_df' not in globals():
    raise NameError('labeled_df not defined. Run all previous cells first.')

# Ensure output paths are defined
if 'FINAL_OUTPUT' not in globals():
    _data_dir = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    FINAL_OUTPUT = str(_data_dir / 'processed' / 'labeled_contracts.csv')
    Path(FINAL_OUTPUT).parent.mkdir(parents=True, exist_ok=True)

# Save final labeled dataset
labeled_df.to_csv(FINAL_OUTPUT, index=False)
print(f'✓ Saved {len(labeled_df):,} labeled contracts')
print(f'  {FINAL_OUTPUT}')

✓ Saved 3,633 labeled contracts
  C:\Users\sarme\IS392Final\data\processed\labeled_contracts.csv


### 8.1 Generate Dataset Summary Report

In [15]:
# Build summary report
def _pct(numerator, denominator):
    return (numerator / denominator * 100) if denominator else 0.0

# Verify required data
if 'run_metadata' not in globals():
    raise NameError('run_metadata not defined. Run all previous cells first.')

if 'SUMMARY_FILE' not in globals() or 'INTERIM_OUTPUT' not in globals() or 'FINAL_OUTPUT' not in globals():
    from pathlib import Path
    _data_dir = Path(r'C:/Users/sarme/IS392Final/data').resolve()
    INTERIM_OUTPUT = globals().get('INTERIM_OUTPUT', str(_data_dir / 'interim' / 'filtered_physical_deliverables.csv'))
    FINAL_OUTPUT = globals().get('FINAL_OUTPUT', str(_data_dir / 'processed' / 'labeled_contracts.csv'))
    SUMMARY_FILE = globals().get('SUMMARY_FILE', str(_data_dir / 'processed' / 'dataset_summary.txt'))

# Extract metadata
total_loaded = run_metadata.get('total_rows_loaded', 0)
psc_kept = run_metadata.get('total_rows_after_psc_check', 0)
value_kept = run_metadata.get('total_rows_after_value_check', 0)
final_count = run_metadata.get('final_labeled_count', 0)
high_value_pos = run_metadata.get('high_value_positives', 0)
competitive_pos = run_metadata.get('competitive_positives', 0)
run_date = run_metadata.get('run_date', 'N/A')

# Build report
summary_lines = [
    '=' * 70,
    'DATASET SUMMARY REPORT - USAspending CSV Files',
    '=' * 70,
    f"Run Date: {run_date}",
    f"Data Source: C:/Users/sarme/IS392Final/data",
    '',
    '-' * 70,
    'DATA LOADING',
    '-' * 70,
    f"Total rows loaded: {total_loaded:,}",
    '',
    '-' * 70,
    'FILTERING SANITY CHECKS',
    '-' * 70,
    f"After quality check: {psc_kept:,}",
    f"After value filter (>= $500k): {value_kept:,}",
    f"Quality check retention rate: {_pct(psc_kept, total_loaded):.2f}%",
    f"Value filter retention rate: {_pct(value_kept, psc_kept):.2f}%",
    '',
    '-' * 70,
    'SAMPLING',
    '-' * 70,
    f"Unique PIIDs before sampling: {run_metadata.get('unique_piids_before_sample', 0):,}",
    f"Sampled PIIDs: {run_metadata.get('unique_piids_after_sample', 0):,}",
    '',
    '-' * 70,
    'LABEL DISTRIBUTION',
    '-' * 70,
    f"Final labeled contracts: {final_count:,}",
    '',
    f"high_value: {high_value_pos:,} positive ({_pct(high_value_pos, final_count):.2f}%)",
    f"            {final_count - high_value_pos:,} negative ({_pct(final_count - high_value_pos, final_count):.2f}%)",
    '',
    f"competitive: {competitive_pos:,} positive ({_pct(competitive_pos, final_count):.2f}%)",
    f"             {final_count - competitive_pos:,} negative ({_pct(final_count - competitive_pos, final_count):.2f}%)",
    '',
    '-' * 70,
    'OUTPUT FILES',
    '-' * 70,
    f"Intermediate: {INTERIM_OUTPUT}",
    f"Final:        {FINAL_OUTPUT}",
    '=' * 70,
]

# Write summary
summary_text = '\n'.join(summary_lines)
with open(SUMMARY_FILE, 'w', encoding='utf-8') as f:
    f.write(summary_text)

# Print summary
print(summary_text)
print(f'\n✓ Summary saved: {SUMMARY_FILE}')

DATASET SUMMARY REPORT - USAspending CSV Files
Run Date: 2026-04-27 13:51:08
Data Source: C:/Users/sarme/IS392Final/data

----------------------------------------------------------------------
DATA LOADING
----------------------------------------------------------------------
Total rows loaded: 100,000

----------------------------------------------------------------------
FILTERING SANITY CHECKS
----------------------------------------------------------------------
After quality check: 76,169
After value filter (>= $500k): 3,733
Quality check retention rate: 76.17%
Value filter retention rate: 4.90%

----------------------------------------------------------------------
SAMPLING
----------------------------------------------------------------------
Unique PIIDs before sampling: 3,633
Sampled PIIDs: 3,633

----------------------------------------------------------------------
LABEL DISTRIBUTION
----------------------------------------------------------------------
Final labeled contrac

## 9. Next Steps

This notebook produces the foundation dataset for all downstream analysis. Next:

1. **Run `05_text_preprocessing_lda.ipynb`** — Apply two-track text preprocessing
2. **Run `06_classification.ipynb`** — Build feature matrices, train classification models
3. **(Optional) Run `07_gao_validation.ipynb`** — Cross-validate with GAO data

**Key outputs:**
- `data/processed/labeled_contracts.csv` — Complete labeled dataset
- `data/processed/dataset_summary.txt` — Metadata and class balance summary